# Logistic Regression for University Admission Prediction

This notebook builds a logistic regression model that predicts whether a student is admitted to university based on two exam scores. We'll follow four steps: explore and visualize the data, fit a logistic regression model with scikit-learn, generate predictions, and evaluate the model.

**Dataset:** `ex2data1.txt` — each row contains two exam scores and an admission label (1 = admitted, 0 = not admitted).

## 1. Data Exploration

First, we load the dataset with pandas and look at its shape and first few rows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load the dataset (no header in the file, so we name the columns ourselves)
col_names = ['Exam1', 'Exam2', 'Admitted']
df = pd.read_csv('ex2data1.txt', header=None, names=col_names)

print("Shape:", df.shape)
df.head()

In [ ]:
# Quick summary stats and class balance
print(df.describe())
print()
print("Admission counts:")
print(df['Admitted'].value_counts())

### Scatter plot of the raw data

Each point is a student, plotted by their two exam scores. Green circles are admitted students, red crosses are students who were not admitted. Visually, admitted students cluster toward higher scores on both exams, with a fairly clear (though not perfect) separation between the two groups.

In [ ]:
admitted = df[df['Admitted'] == 1]
not_admitted = df[df['Admitted'] == 0]

plt.figure(figsize=(8, 6))
plt.scatter(admitted['Exam1'], admitted['Exam2'], c='green', marker='o', label='Admitted')
plt.scatter(not_admitted['Exam1'], not_admitted['Exam2'], c='red', marker='x', label='Not Admitted')
plt.xlabel('Exam 1 Score')
plt.ylabel('Exam 2 Score')
plt.title('Student Admission Data')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. Applying Logistic Regression with scikit-learn

We split the dataframe into features `X` (the two exam scores) and target `y` (the admission label), then fit a `LogisticRegression` model. We don't need to scale the features here since both exam scores are already on a similar 0–100 scale, and we're using the full dataset to train (matching the original exercise, which evaluates on the same data it trains on).

In [ ]:
X = df[['Exam1', 'Exam2']].values
y = df['Admitted'].values

model = LogisticRegression()
model.fit(X, y)

print("Learned coefficients (theta1, theta2):", model.coef_[0])
print("Learned intercept (theta0):", model.intercept_[0])

## 3. Making Predictions

With the model trained, we predict admission labels for every student in the dataset and compare them to the true labels to calculate accuracy.

In [ ]:
y_pred = model.predict(X)

accuracy = accuracy_score(y, y_pred)
print(f"Training accuracy: {accuracy:.2%}")

In [ ]:
# Example: predict admission probability for a student who scored 45 on Exam 1 and 85 on Exam 2
sample_scores = np.array([[45, 85]])
sample_prob = model.predict_proba(sample_scores)[0, 1]
sample_pred = model.predict(sample_scores)[0]

print(f"Student with scores (45, 85): probability of admission = {sample_prob:.4f}")
print(f"Predicted class: {'Admitted' if sample_pred == 1 else 'Not Admitted'}")

## 4. Model Evaluation

We look at the confusion matrix and classification report to understand where the model gets things right and wrong, then plot the decision boundary it learned.

In [ ]:
cm = confusion_matrix(y, y_pred)
print("Confusion matrix:")
print(cm)
print()
print("Classification report:")
print(classification_report(y, y_pred, target_names=['Not Admitted', 'Admitted']))

### Decision boundary

Logistic regression learns a linear boundary in the feature space. Points on one side are classified as admitted, points on the other side as not admitted. We compute this line from the model's coefficients: it's the set of points where the predicted probability equals 0.5, i.e. where `theta0 + theta1*Exam1 + theta2*Exam2 = 0`.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(admitted['Exam1'], admitted['Exam2'], c='green', marker='o', label='Admitted')
plt.scatter(not_admitted['Exam1'], not_admitted['Exam2'], c='red', marker='x', label='Not Admitted')

# Solve for Exam2 in terms of Exam1 along the decision boundary
x1_vals = np.linspace(X[:, 0].min() - 2, X[:, 0].max() + 2, 100)
coef = model.coef_[0]
intercept = model.intercept_[0]
x2_vals = -(coef[0] * x1_vals + intercept) / coef[1]

plt.plot(x1_vals, x2_vals, color='blue', label='Decision Boundary')
plt.xlabel('Exam 1 Score')
plt.ylabel('Exam 2 Score')
plt.title('Logistic Regression Decision Boundary')
plt.legend()
plt.ylim(X[:, 1].min() - 5, X[:, 1].max() + 5)
plt.grid(alpha=0.3)
plt.show()

### Interpretation

**Model fit:** The trained model achieves about **89% training accuracy** on this dataset of 100 students (60 admitted, 40 not admitted), correctly classifying the large majority of cases. The confusion matrix shows the errors are fairly balanced between the two classes rather than the model favoring one outcome.

**Coefficients:** Both exam scores have positive coefficients of similar magnitude, meaning higher scores on either exam increase the predicted probability of admission, and the two exams contribute roughly equally to that prediction. The decision boundary is a straight line because logistic regression is a linear classifier in the feature space — it can only separate the two classes with a straight line (or, with more features, a flat hyperplane).

**Where it struggles:** Looking at the scatter plot, most misclassifications happen near the boundary itself, where admitted and not-admitted students have similar score combinations. This is expected: a few students close to the boundary scored well but weren't admitted, or vice versa, and a purely linear model can't capture that overlap perfectly.

**Caveat on accuracy:** This accuracy is measured on the same data used to train the model (no train/test split), so it reflects how well the model fits the data it has already seen rather than how it would generalize to new, unseen students. For a more rigorous estimate of real-world performance, the next step would be to split the data into training and test sets, or use cross-validation.